In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# Imports
import inspect
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import cv2


In [ ]:
# Load the trained model
MODEL_PATH = "/content/drive/MyDrive/brain-tumor-classification/models/best_model.keras"

model = tf.keras.models.load_model(MODEL_PATH)
model.trainable = False

if "class_names" not in globals():
    class_names = ["glioma", "meningioma", "notumor", "pituitary"]

print("Model loaded.")
print("Input shape :", model.input_shape)
print("Output shape:", model.output_shape)
print("Classes     :", class_names)


In [ ]:
# Debug cell: inspect top-level and backbone layers
for i, layer in enumerate(model.layers):
    print(f"{i:02d} | {layer.name:35s} | {layer.__class__.__name__}")

print("\nNested model layers:\n")
for layer in model.layers:
    if isinstance(layer, tf.keras.Model):
        print("Nested model:", layer.name)
        for sub in layer.layers[-20:]:
            print("   ", sub.name, "|", sub.__class__.__name__)
        print("-" * 70)


In [ ]:
# Helper functions for robust Grad-CAM on loaded .keras models
CONV_LAYER_TYPES = (
    tf.keras.layers.Conv2D,
    tf.keras.layers.DepthwiseConv2D,
    tf.keras.layers.SeparableConv2D,
    tf.keras.layers.Conv2DTranspose,
)

def supports_training_arg(layer):
    try:
        return "training" in inspect.signature(layer.call).parameters
    except Exception:
        return False

def safe_call(layer, x):
    if supports_training_arg(layer):
        return layer(x, training=False)
    return layer(x)

def normalize_map(cam):
    cam = np.array(cam, dtype=np.float32)
    cam = cam - np.min(cam)
    denom = np.max(cam)
    if denom < 1e-8:
        return np.zeros_like(cam, dtype=np.float32)
    return cam / denom

def find_backbone_model(full_model):
    nested_models = [layer for layer in full_model.layers if isinstance(layer, tf.keras.Model)]
    if not nested_models:
        raise ValueError("No nested backbone model found in the loaded model.")
    return max(nested_models, key=lambda m: sum(isinstance(layer, CONV_LAYER_TYPES) for layer in m.layers))

def get_candidate_target_layers(backbone):
    preferred = [
        "top_activation",
        "top_conv",
        "block7a_project_bn",
        "block7a_project_conv",
        "block6d_project_bn",
        "block6d_project_conv",
    ]

    candidate_names = []
    seen = set()

    for name in preferred:
        try:
            layer = backbone.get_layer(name)
            if len(layer.output.shape) == 4 and name not in seen:
                candidate_names.append(name)
                seen.add(name)
        except Exception:
            pass

    for layer in reversed(backbone.layers):
        try:
            if len(layer.output.shape) == 4 and layer.name not in seen:
                candidate_names.append(layer.name)
                seen.add(layer.name)
        except Exception:
            pass

    if not candidate_names:
        raise ValueError("No 4D candidate layers found for Grad-CAM.")

    return candidate_names

def build_exact_gradcam_models(full_model, target_layer_name):
    backbone = find_backbone_model(full_model)
    backbone_index = full_model.layers.index(backbone)

    target_index = None
    for i, layer in enumerate(backbone.layers):
        if layer.name == target_layer_name:
            target_index = i
            break

    if target_index is None:
        raise ValueError(f"Layer '{target_layer_name}' was not found in backbone '{backbone.name}'.")

    target_layer = backbone.get_layer(target_layer_name)
    if len(target_layer.output.shape) != 4:
        raise ValueError(f"Layer '{target_layer_name}' is not a 4D feature map layer.")

    model_input = tf.keras.Input(shape=full_model.input_shape[1:], name="gradcam_model_input")
    x = model_input
    for layer in full_model.layers[1:backbone_index]:
        x = safe_call(layer, x)
    pre_backbone_model = tf.keras.Model(model_input, x, name=f"pre_backbone_{target_layer_name}")

    backbone_input = tf.keras.Input(shape=backbone.input_shape[1:], name="gradcam_backbone_input")
    y = backbone_input
    for layer in backbone.layers[1:target_index + 1]:
        y = safe_call(layer, y)
    pre_target_model = tf.keras.Model(backbone_input, y, name=f"pre_target_{target_layer_name}")

    target_input = tf.keras.Input(shape=target_layer.output.shape[1:], name="gradcam_target_input")
    z = target_input
    for layer in backbone.layers[target_index + 1:]:
        z = safe_call(layer, z)
    for layer in full_model.layers[backbone_index + 1:]:
        z = safe_call(layer, z)
    post_target_model = tf.keras.Model(target_input, z, name=f"post_target_{target_layer_name}")

    return {
        "backbone": backbone,
        "target_layer_name": target_layer_name,
        "pre_backbone_model": pre_backbone_model,
        "pre_target_model": pre_target_model,
        "post_target_model": post_target_model,
    }


In [ ]:
# Image loading, prediction, and Grad-CAM generation
IMG_SIZE = tuple(model.input_shape[1:3])
backbone = find_backbone_model(model)
candidate_target_layers = get_candidate_target_layers(backbone)

print("Backbone model:", backbone.name)
print("Candidate target layers:", candidate_target_layers[:10])

def load_image_for_model(img_path, target_size=IMG_SIZE):
    img = tf.keras.utils.load_img(img_path, target_size=target_size)
    img_array = tf.keras.utils.img_to_array(img).astype("float32")
    batch = np.expand_dims(img_array, axis=0)
    return img, img_array, batch

def predict_image(model, img_path, class_names):
    pil_img, img_array, batch = load_image_for_model(img_path)
    preds = model.predict(batch, verbose=0)
    pred_index = int(np.argmax(preds[0]))
    confidence = float(preds[0][pred_index])
    pred_class = class_names[pred_index]
    return {
        "pil_img": pil_img,
        "img_array": img_array,
        "batch": batch,
        "preds": preds,
        "pred_index": pred_index,
        "pred_class": pred_class,
        "confidence": confidence,
    }

def compute_gradcam_for_layer(img_batch, full_model, target_layer_name, class_index=None, debug=False):
    parts = build_exact_gradcam_models(full_model, target_layer_name)

    backbone_input = parts["pre_backbone_model"](img_batch, training=False)

    with tf.GradientTape() as tape:
        target_activations = parts["pre_target_model"](backbone_input, training=False)
        tape.watch(target_activations)
        preds = parts["post_target_model"](target_activations, training=False)

        if class_index is None:
            class_index = tf.argmax(preds[0])

        class_score = preds[:, class_index]

    grads = tape.gradient(class_score, target_activations)
    if grads is None:
        raise ValueError(f"Gradients are None for layer '{target_layer_name}'.")

    activations = target_activations[0].numpy()
    gradients = grads[0].numpy()

    weights = np.mean(gradients, axis=(0, 1))
    cam_raw = np.sum(activations * weights[np.newaxis, np.newaxis, :], axis=-1)
    cam_relu = np.maximum(cam_raw, 0)

    heatmap = normalize_map(cam_relu)
    used_absolute_fallback = False

    if float(np.max(heatmap)) < 1e-8:
        heatmap = normalize_map(np.abs(cam_raw))
        used_absolute_fallback = True

    if debug:
        print("Target layer          :", target_layer_name)
        print("backbone_input shape  :", backbone_input.shape)
        print("target_activations    :", target_activations.shape)
        print("preds shape           :", preds.shape)
        print("grads shape           :", grads.shape)
        print("heatmap max           :", float(np.max(heatmap)))
        print("absolute fallback used:", used_absolute_fallback)

    return {
        "heatmap": heatmap,
        "preds": preds.numpy(),
        "target_layer_name": target_layer_name,
        "used_absolute_fallback": used_absolute_fallback,
    }

def make_gradcam_heatmap(img_batch, full_model, class_index=None, debug=True):
    errors = []
    best_result = None
    best_score = -1.0

    for layer_name in candidate_target_layers:
        try:
            result = compute_gradcam_for_layer(
                img_batch=img_batch,
                full_model=full_model,
                target_layer_name=layer_name,
                class_index=class_index,
                debug=debug,
            )

            score = float(np.max(result["heatmap"]))
            if score > best_score:
                best_score = score
                best_result = result

            if score > 0.20 and not result["used_absolute_fallback"]:
                return result

        except Exception as exc:
            errors.append(f"{layer_name}: {exc}")

    if best_result is not None:
        print("Using best available target layer:", best_result["target_layer_name"])
        return best_result

    raise ValueError("All Grad-CAM candidate layers failed:\n" + "\n".join(errors))


In [ ]:
# Overlay helper and final display function
def overlay_gradcam(img_array, heatmap, alpha=0.35):
    img_uint8 = np.uint8(np.clip(img_array, 0, 255))

    if heatmap is None or np.size(heatmap) == 0:
        raise ValueError("Heatmap is empty before resize.")

    if heatmap.ndim != 2:
        raise ValueError(f"Heatmap must be 2D, got shape {heatmap.shape}")

    heatmap_resized = cv2.resize(heatmap.astype(np.float32), (img_uint8.shape[1], img_uint8.shape[0]))
    heatmap_uint8 = np.uint8(255 * heatmap_resized)
    heatmap_color = cv2.applyColorMap(heatmap_uint8, cv2.COLORMAP_JET)
    heatmap_color = cv2.cvtColor(heatmap_color, cv2.COLOR_BGR2RGB)

    overlay = cv2.addWeighted(img_uint8, 1 - alpha, heatmap_color, alpha, 0)
    return heatmap_resized, heatmap_color, overlay

def show_prediction_and_gradcam(img_path, model, class_names, alpha=0.35, debug=True):
    result = predict_image(model, img_path, class_names)

    cam_result = make_gradcam_heatmap(
        img_batch=result["batch"],
        full_model=model,
        class_index=result["pred_index"],
        debug=debug,
    )

    heatmap_resized, heatmap_color, overlay = overlay_gradcam(
        result["img_array"],
        cam_result["heatmap"],
        alpha=alpha,
    )

    plt.figure(figsize=(16, 5))

    plt.subplot(1, 3, 1)
    plt.imshow(np.uint8(result["img_array"]))
    plt.title("Original MRI")
    plt.axis("off")

    plt.subplot(1, 3, 2)
    plt.imshow(heatmap_resized, cmap="jet")
    plt.title(f"Grad-CAM ({cam_result['target_layer_name']})")
    plt.axis("off")

    plt.subplot(1, 3, 3)
    plt.imshow(overlay)
    plt.title(f"{result['pred_class']} ({result['confidence']:.2%})")
    plt.axis("off")

    plt.tight_layout()
    plt.show()

    print("Predicted class:", result["pred_class"])
    print("Confidence     :", f"{result['confidence']:.2%}")
    print("Target layer   :", cam_result["target_layer_name"])
    print("Fallback used  :", cam_result["used_absolute_fallback"])

    return {
        **result,
        **cam_result,
        "heatmap_resized": heatmap_resized,
        "overlay": overlay,
    }


In [ ]:
# Final test cell
# Change img_path to your MRI image file in Colab.
img_path = "/content/download.jpg"

output = show_prediction_and_gradcam(
    img_path=img_path,
    model=model,
    class_names=class_names,
    alpha=0.35,
    debug=True,
)
